In [9]:
# For single-file datasets like WELFake
df = pd.read_csv('data/WELFake_Dataset.csv')
df = df.dropna(subset=['text']) # remove empty rows
print(f"Loaded {len(df)} articles successfully!")

Loaded 72095 articles successfully!


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Clean missing text values (if any)
df['text'] = df['text'].fillna('')

# 2. Define inputs (X) and targets (y)
X = df['text']
y = df['label']

# 3. Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Convert text to numerical TF-IDF vectors
print("Vectorizing text...")
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7, max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. Train Logistic Regression Model
print("Training the Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# 6. Evaluate accuracy
predictions = model.predict(X_test_vec)
acc = accuracy_score(y_test, predictions)

print(f"\n✅ Training Complete!")
print(f"Model Accuracy: {acc * 100:.2f}%\n")
print("Detailed Classification Report:\n", classification_report(y_test, predictions))

Vectorizing text...
Training the Logistic Regression model...

✅ Training Complete!
Model Accuracy: 93.86%

Detailed Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.93      0.94      7010
           1       0.94      0.95      0.94      7409

    accuracy                           0.94     14419
   macro avg       0.94      0.94      0.94     14419
weighted avg       0.94      0.94      0.94     14419



In [12]:
def predict_news(news_text):
    # Convert input text using the trained vectorizer
    text_vec = vectorizer.transform([news_text])
    
    # Predict class and confidence probability
    prediction = model.predict(text_vec)[0]
    prob = model.predict_proba(text_vec)[0]
    
    # Format output based on prediction (assuming 1 = Real, 0 = Fake)
    if prediction == 1:
        print(f"Prediction: REAL NEWS (Confidence: {prob[1]*100:.1f}%)")
    else:
        print(f"Prediction: FAKE NEWS (Confidence: {prob[0]*100:.1f}%)")

# --- TEST CUSTOM EXAMPLES ---
print("--- Test 1 ---")
predict_news("BREAKING: NASA confirms discovery of alien spaceships near Jupiter rings!")

print("\n--- Test 2 ---")
predict_news("The Federal Reserve announced an interest rate decision following its quarterly meeting today.")

--- Test 1 ---
Prediction: REAL NEWS (Confidence: 96.7%)

--- Test 2 ---
Prediction: REAL NEWS (Confidence: 96.2%)
